### 1. HotPepperから兵庫県の5000件の飲食店データを取得している.
- *AllRestaurants.json*に格納

In [1]:
import requests
import json

BASE_URL = "https://webservice.recruit.co.jp/hotpepper/gourmet/v1/"
API_KEY = "073c5f34bcea3eb9"
LARGE_AREA = "Z024"
COUNT = 100
MAX_RESULTS = 5000

shops_data = []

for start in range(1, MAX_RESULTS + 1, COUNT):
    params = {
        "key": API_KEY,
        "large_area": LARGE_AREA,
        "count": COUNT,
        "start": start,
        "format": "json"
    }

    response = requests.get(BASE_URL, params=params)
    response_data = response.json()

    for shop in response_data['results']['shop']:
        extracted_data = {
            "name": shop['name'],
            "genre_name": shop['genre']['name'],
            "budget_average": shop['budget']['name'],
            "private_room": shop['private_room'],
            "non_smoking": shop['non_smoking'],
            "wifi": shop['wifi'],
            "card": shop['card'],
            "parking": shop['parking']
        }
        shops_data.append(extracted_data)

# 5000件分のデータをJSONファイルに出力
with open("AllRestaurants", "w") as f:
    json.dump(shops_data, f, ensure_ascii=False, indent=4)

print("Data extraction completed!")

Data extraction completed!


### 2. 取得した5000件のデータから「低価格」「中価格」「高価格」と3分割する閾値を調べる

In [5]:
with open('AllRestaurants.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

budgets_int = []
for shop in data:
    budget = shop['budget_average']
    if "～" in budget:
        low, high = budget.split("～")
        low = low.replace("円", "").replace(",", "").strip()
        high = high.split("円")[0].replace(",", "").strip()
        
        # low と high が数字のみで構成されているかを確認
        if low.isdigit() and high.isdigit():
            avg = (int(low) + int(high)) // 2
            budgets_int.append(avg)
    elif budget.replace("円", "").replace(",", "").strip().isdigit():
        budgets_int.append(int(budget.replace("円", "").replace(",", "").strip()))

budgets_int.sort()
#閾値1
low_threshold = budgets_int[len(budgets_int) // 3]
#閾値2
middle_threshold = budgets_int[2 * len(budgets_int) // 3]

print(f"低価格帯の最大値: {low_threshold}円")
print(f"中価格帯の最大値: {middle_threshold}円")
print("高価格帯はそれ以上")

低価格帯の最大値: 2500円
中価格帯の最大値: 3500円
高価格帯はそれ以上


### 3. 2を元にアルゴリズム評価用のデータを100件選定する
- ジャンルと価格帯の組み合わせをできるだけ網羅するように選択している
- *SelectedRestaurants.json*に格納

In [7]:
import random

with open('AllRestaurants.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

selected_shops = []

for shop in data:
    budget_str = shop['budget_average']
    if "～" in budget_str:
        limits = budget_str.replace('円', '').split('～')
        lower_limit = int(limits[0]) if limits[0].isdigit() else None
        upper_limit = int(limits[1]) if limits[1].isdigit() else None
        if lower_limit and upper_limit:
            shop['budget_int'] = (lower_limit + upper_limit) // 2

genre_list = ["中華", "居酒屋", "ダイニングバー・バル", "和食", "お好み焼き・もんじゃ",
              "韓国料理", "イタリアン・フレンチ", "焼肉・ホルモン", "洋食", "アジア・エスニック料理", 
              "創作料理", "各国料理", "ラーメン"]

for genre in genre_list:
    shops_in_genre = [shop for shop in data if shop['genre_name'] == genre and 'budget_int' in shop]
    
    low_budget_shops = [shop for shop in shops_in_genre if shop['budget_int'] <= low_threshold]
    middle_budget_shops = [shop for shop in shops_in_genre if low_threshold < shop['budget_int'] <= middle_threshold]
    high_budget_shops = [shop for shop in shops_in_genre if shop['budget_int'] > middle_threshold]
    
    selected_genre_shops = []
    for budget_shops, num in zip([low_budget_shops, middle_budget_shops, high_budget_shops], [3, 2, 2]):
        selected_genre_shops.extend(random.sample(budget_shops, min(num, len(budget_shops))))

    while len(selected_genre_shops) < 7 and (low_budget_shops or middle_budget_shops or high_budget_shops):
        for budget_shops in [low_budget_shops, middle_budget_shops, high_budget_shops]:
            if len(selected_genre_shops) < 7 and budget_shops:
                selected_shop = random.choice(budget_shops)
                selected_genre_shops.append(selected_shop)
                budget_shops.remove(selected_shop)
                
    selected_shops.extend(selected_genre_shops[:7])

add_genres = ["中華", "居酒屋", "ダイニングバー・バル", "和食", "イタリアン・フレンチ", 
              "焼肉・ホルモン", "洋食", "創作料理", "各国料理"]

for genre in add_genres:
    shops_in_genre = [shop for shop in data if shop['genre_name'] == genre and 'budget_int' in shop]
    high_budget_shops = [shop for shop in shops_in_genre if shop['budget_int'] > middle_threshold]
    
    high_budget_shops = [shop for shop in high_budget_shops if shop not in selected_shops]
    
    if high_budget_shops:
        selected_shop = random.choice(high_budget_shops)
        selected_shops.append(selected_shop)

with open('SelectedRestaurants.json', 'w', encoding='utf-8') as f:
    json.dump(selected_shops, f, ensure_ascii=False, indent=4)

### 4. 3で選んだ100件分のデータに手作業で取得した総合評価を加える
- 総合評価はぐるナビで取得
- *SelectedRestaurants+Rate.json*に格納

### 5. 4で作成したテキストデータを微修正
- このデータを用いて, Google Formで評価を取得
- 駐車場の有無と個室の有無はどちらも0, 1で表すため, 「あり」「なし」以外の不要な文言を削除
- *RestaurantsText.json*に格納

In [11]:
#SelectedRestaurants+Rate.jsonの微修正 → RestaurantsText.json(Formで使う)

with open('SelectedRestaurants+Rate.json', 'r') as file:
    shops = json.load(file)

# "parking"と"private_room"の値を修正
for shop in shops:
    shop['parking'] = 'あり' if 'あり' in shop['parking'] else 'なし'
    shop['private_room'] = 'あり' if 'あり' in shop['private_room'] else 'なし'

with open('RestaurantsText.json', 'w', encoding='utf-8') as file:
    json.dump(shops, file, ensure_ascii=False, indent=4)

### 6. 5で修正したテキストデータを数値データに変換
- *RestaurantsNumber.json*に格納

In [32]:
import json
import math

def transform_restaurant_data(input_file, output_file):
    genres = ["中華", "居酒屋", "ダイニングバー・バル", "和食", "お好み焼き・もんじゃ", "韓国料理",
              "イタリアン・フレンチ", "焼肉・ホルモン", "洋食", "アジア・エスニック料理", "創作料理", "各国料理", "ラーメン"]
    
    with open(input_file, 'r', encoding='utf-8') as file:
        data = json.load(file)
        
        for restaurant in data:
            # Remove sub_genre
            if 'sub_genre_name' in restaurant:
                del restaurant['sub_genre_name']
            
            # Genre Encoding
            genre = restaurant['genre_name']
            restaurant['genre_encoding'] = " ".join(map(str, [1 if g == genre else 0 for g in genres]))
            del restaurant['genre_name']
            
            # Convert yes/no attributes to binary
            binary_attributes = [('private_room', ['あり', 'なし']), 
                                 ('wifi', ['あり', 'なし']), 
                                 ('card', ['利用可', '利用不可']), 
                                 ('parking', ['あり', 'なし'])]
                                 
            for attr, allowed_values in binary_attributes:
                value = restaurant[attr]
                assert value in allowed_values, f"Invalid value '{value}' for attribute '{attr}'"
                restaurant[attr] = 1 if value == 'あり' or value == '利用可' else 0
            
            # Non-smoking
            ns = restaurant['non_smoking']
            assert ns in ['禁煙席なし', '一部禁煙', '禁煙席あり', '全面禁煙'], f"Invalid value '{ns}' for attribute 'non_smoking'"
            restaurant['non_smoking'] = 1 if ns == '禁煙席なし' else 0.5 if ns in ['一部禁煙', '禁煙席あり'] else 0
            
             # Budget Average
            budget_str = restaurant['budget_average']
            budget_values = [int(val) for val in budget_str.replace("円", "").split("～") if val.isdigit()]
            if budget_values:
                lower_bound = budget_values[0]
                upper_bound = budget_values[1] if len(budget_values) > 1 else lower_bound
                restaurant['budget_average'] = math.floor(lower_bound + (upper_bound - lower_bound) * 0.5)
            else:
                restaurant['budget_average'] = None

            
    # Save transformed data to a new json file
    with open(output_file, 'w', encoding='utf-8') as file:
        json.dump(data, file, ensure_ascii=False, indent=4)

# Example usage
transform_restaurant_data('RestaurantsText.json', 'RestaurantsNumber.json')

### 7. 6で作成した数値データを100個のリストに変換する
- 決定木の入力に用いる
- *Features.json*に格納

In [37]:

def transform_restaurant_data(input_file, output_file):
    genres = ["中華", "居酒屋", "ダイニングバー・バル", "和食", "お好み焼き・もんじゃ", "韓国料理",
              "イタリアン・フレンチ", "焼肉・ホルモン", "洋食", "アジア・エスニック料理", "創作料理", "各国料理", "ラーメン"]
    
    with open(input_file, 'r', encoding='utf-8') as file:
        data = json.load(file)
        transformed_data = []
        
        for restaurant in data:
            # Your previous transformations here...
            
            # Creating a single list of all features for each restaurant
            features = []
            
            # Adding genre_encoding as individual elements
            genre_encoding_str = restaurant['genre_encoding']
            genre_encoding = [int(x) for x in genre_encoding_str.split()]
            features.extend(genre_encoding)
            
            # Adding other features
            other_features = [
                restaurant['budget_average'], 
                restaurant['private_room'], 
                restaurant['non_smoking'], 
                restaurant['wifi'], 
                restaurant['card'], 
                restaurant['parking'], 
                restaurant['rating']
            ]
            features.extend(other_features)
            
            transformed_data.append(features)
            
    # Save transformed data to a new json file, writing each restaurant's data in a new line
    with open(output_file, 'w', encoding='utf-8') as file:
        file.write("[\n")
        for i, restaurant in enumerate(transformed_data):
            json.dump(restaurant, file, ensure_ascii=False)
            if i < len(transformed_data) - 1:
                file.write(",\n")
            else:
                file.write("\n")
        file.write("]\n")

transform_restaurant_data('RestaurantsNumber.json', 'Features.json')
